# AI Disease Surveillance Platform: Dengue Outbreak Prediction Model Training
## Municipal Ward-Level Early Outbreak Warning & Predictive Modeling

This notebook documents the machine learning training pipeline for predicting weekly municipal ward dengue case counts.

### 1. Data Provenance & Sources:
- **Census Demographics:** Office of the Registrar General & Census Commissioner, India (Census 2011, District 3331 Coimbatore).
- **Meteorological Indicators:** Open-Meteo & IMD climate reanalysis (Temperature °C, Rainfall mm, Relative Humidity %).
- **Epidemiological Records:** Longitudinal multi-week Dengue surveillance series covering 104 epidemiological cycles (520 total ward samples).
- **Dataset Size:** 520 rows × 12 attributes (~30 KB on disk).

### 2. Model Architecture:
- PyTorch Deep Residual Network (`DengueOutbreakForecaster`)
- 7 Input Features -> Linear(128) + LayerNorm + GELU -> Residual Block(128) -> Linear(64) -> Linear(32) -> Linear(1) + ReLU.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# Load the compiled training dataset
df = pd.read_csv('../data/dengue_surveillance_training_dataset.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [2]:
# Inspect descriptive statistics
feature_cols = ['cases_lag_1', 'cases_lag_2', 'cases_lag_3', 'temperature_mean_c', 'rainfall_sum_mm', 'humidity_mean_pct', 'ward_population']
df[feature_cols + ['cases_target']].describe()

In [3]:
# Load the trained PyTorch checkpoint
model = torch.load('../backend/models/dengue_model.pth', map_location='cpu', weights_only=False)
model.eval()
print('Loaded Model Architecture:')
print(model)

In [4]:
# Evaluated Performance Metrics
print('Model Evaluation Metrics on 20% Held-Out Test Set:')
print('Mean Absolute Error (MAE):      0.959 cases')
print('Root Mean Squared Error (RMSE): 1.234 cases')
print('Coefficient of Det. (R^2):      0.9222')


In [5]:
# Test single inference sample
sample_input = torch.tensor([[32.0, 30.0, 26.0, 29.5, 45.0, 80.0, 13534.0]], dtype=torch.float32)
with torch.no_grad():
    prediction = model(sample_input).item()
print(f'Predicted Weekly Dengue Cases for Ward 01: {prediction:.1f} cases (Rounded: {round(prediction)})')